# Error analysis and robustness #
Material for the report section *Error Analysis and Robustness*. Combines the saved test forecasts of every model that
has been run (`xgboost.ipynb`, `neural_network.ipynb`) on the same 10,495 test hours and breaks the errors down by price
area, hour, week, temperature and over time. Feature importance and temperature-noise sensitivity are computed in the model
notebooks (they need the trained models) and only plotted here. The test period (20 Jan - 3 Apr 2023) has no public
holidays, so holidays are scored on the validation windows (4 Dec - 14 Jan).

Figures are saved to `figures/` as PDF for `\includegraphics`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from forecast_checks import ForecastValidator, MakeTrainingData, load_tables

RESULTS, FIGURES = Path("results"), Path("figures")
FIGURES.mkdir(exist_ok=True)
MODELS = {"XGBoost": "xgboost", "Neural network": "neural_network"}
AREAS = {"oslo": "NO1 Oslo", "stavanger": "NO2 Stavanger", "trondheim": "NO3 Trondheim", "tromsø": "NO4 Tromsø",
         "bergen": "NO5 Bergen", "helsingfors": "FI Helsingfors"}


def combine(suffix: str) -> ForecastValidator:
    """One validator with the baseline and every model whose results/<model><suffix>.json exists."""
    paths = {name: RESULTS / f"{stem}{suffix}.json" for name, stem in MODELS.items()}
    missing = [name for name, path in paths.items() if not path.exists()]
    if missing:
        print(f"Not run yet, left out: {missing}")
    loaded = {name: ForecastValidator.load_predictions(path) for name, path in paths.items() if path.exists()}
    if not loaded:
        raise FileNotFoundError(f"No results/*{suffix}.json; run the model notebooks first")
    combined = next(iter(loaded.values()))
    for name, validator in loaded.items():
        if name not in combined.forecasts:
            combined.add(name, validator.forecasts[name])   #add checks that the models were scored on the same hours
    combined.groups = combined.groups.map(AREAS).rename("Price area")
    return combined


test = combine("_predictions")
df = MakeTrainingData().df
info = df.loc[test.actual.index]
assert (info["location"].astype(str).map(AREAS) == test.groups).all()
print(f"{len(info)} test hours, {info['date'].min().date()} - {info['date'].max().date()}")
test.summary(skill=True)

## Per price area (table) ##
skill = 1 - MAE / MAE of the baseline, in %: above 0 is better than *same hour last week*. nMAE and bias in % of
consumption make areas of different size comparable.

In [ ]:
per_area = test.per_group(skill=True)
overall = test.summary(skill=True).rename_axis("model").assign(group="All").set_index("group", append=True)
per_area_table = pd.concat([per_area, overall]).assign(skill=lambda t: 100 * t["skill"])
per_area_table = per_area_table[["mae", "nmae", "bias", "skill"]].unstack("model").reindex([*AREAS.values(), "All"])
per_area_table = per_area_table.reindex(columns=list(test.forecasts), level="model")
per_area_table.round(2)

In [ ]:
print(per_area_table.to_latex(float_format="%.2f"))

## By hour of day and by week ##
nMAE rather than MAE, so Oslo (about 6 times the consumption of Tromsø) does not dominate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True, layout="constrained")
test.plot_metric_by(info["hour"].rename("Hour of day (local time)"), ax=axes[0])
#Weeks from the first test day rather than months, as the test period runs from 20 January to 3 April.
#The days left over at the end (fewer than 7) are counted in the last full week
first_day, n_weeks = info["date"].min(), info["date"].nunique() // 7
week = ((info["date"] - first_day).dt.days // 7).clip(upper=n_weeks - 1)
week_start = (first_day + pd.to_timedelta(7 * week, unit="D")).dt.strftime("%d %b")
label = week_start.where(week < n_weeks - 1, week_start + " +")
#Ordered by week, not alphabetically
label = pd.Series(pd.Categorical(label, categories=label.groupby(week).first()), index=label.index, name="Week starting")
test.plot_metric_by(label, ax=axes[1])
axes[1].tick_params(axis="x", labelrotation=45)
fig.savefig(FIGURES / "error_by_hour_and_week.pdf")

## Cold spells ##
Days are ranked by their mean temperature within each location (decile 1 = the coldest 10 % of test days there), so cold
spells in all areas are compared despite their different climates.

In [ ]:
daily_temperature = info.groupby(["location", "date"], observed=True)["temperature"].transform("mean")
decile = (daily_temperature.groupby(info["location"], observed=True)
          .transform(lambda t: pd.qcut(t.rank(method="dense"), 10, labels=False) + 1)
          .rename("Temperature decile (1 = coldest days)"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
test.plot_metric_by(decile, ax=axes[0])
test.plot_metric_by(decile, metric="skill", no_baseline=True, ax=axes[1])
axes[1].axhline(0, color="#52514e", linewidth=0.8)
fig.savefig(FIGURES / "error_by_temperature.pdf")

In [ ]:
cold_spell = (decile == 1).map({True: "Coldest 10 % of days", False: "Other days"}).rename("period")
test.per(cold_spell, skill=True)

## Bias over time: portfolio level shifts ##
A curve that stays away from 0 for weeks means the level of the area has changed (e.g. customers gained or lost), which
consumption lags that are at least 6 days old cannot catch up with immediately.

In [ ]:
fig = test.plot_rolling_bias(info["time"])
fig.savefig(FIGURES / "rolling_bias.pdf")

## Feature importance and temperature forecast error ##
Left: increase in test MAE when a group of inputs is shuffled within each location. Right: increase in test MAE when a
day-ahead temperature forecast error with the given standard deviation is added (our temperature is reanalysis data,
i.e. close to the real weather).

In [ ]:
robustness = {name: load_tables(RESULTS / f"{stem}_robustness.json") for name, stem in MODELS.items()
              if (RESULTS / f"{stem}_robustness.json").exists()}
importance = pd.DataFrame({name: tables["importance"]["relative_increase"] for name, tables in robustness.items()})
temperature_noise = pd.DataFrame({name: tables["temperature_noise"]["relative_increase"] for name, tables in robustness.items()})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
importance.plot.barh(ax=axes[0], color=test.colors(importance.columns))
axes[0].set_xlabel("MAE increase when shuffled [%]")
axes[0].set_ylabel("")
temperature_noise.plot(ax=axes[1], marker="o", color=test.colors(temperature_noise.columns))
axes[1].set_xlabel("Std of the temperature forecast error [°C]")
axes[1].set_ylabel("MAE increase [%]")
fig.savefig(FIGURES / "importance_and_temperature_noise.pdf")
pd.concat({"importance [%]": importance, "temperature noise [%]": temperature_noise})

## Holidays (validation windows) ##
The test period has no public holidays, so Christmas and New Year are scored on the validation windows (4 Dec - 14 Jan),
with each model fitted only on data up to 6 days before each window. Note that the neural network chose its
hyperparameters on these windows, so its validation scores are slightly optimistic compared with XGBoost's.

In [ ]:
val = combine("_validation_predictions")
val_info = df.loc[val.actual.index]
CHRISTMAS = ("2022-12-24", "2023-01-01")
period = (val_info["date"].between(*CHRISTMAS)
          .map({True: "Christmas and New Year (24 Dec - 1 Jan)", False: "Other validation days"}).rename("period"))
val.per(period, skill=True)